In [0]:
# ==========================================================
# dim_region - Type 1 Dimension (Incremental)
# Grain: (country, state, city, postal_code)
# ==========================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
import uuid
from datetime import timedelta

# ----------------------------------------------------------
# Configuration
# ----------------------------------------------------------
bronze_table = "bronze_dev.global_mart_retail.raw_data"
silver_table_dim_region = "silver_dev.global_mart_retail.dim_region"

LOOKBACK_HOURS = 48
BATCH_ID = str(uuid.uuid4())

# ----------------------------------------------------------
# Determine last load timestamp (Incremental)
# ----------------------------------------------------------
if spark.catalog.tableExists(silver_table_dim_region):
    last_loaded_ts = (
        spark.table(silver_table_dim_region)
        .agg(F.max("load_timestamp").alias("max_ts"))
        .collect()[0]["max_ts"]
    )
    if last_loaded_ts:
        last_loaded_ts -= timedelta(hours=LOOKBACK_HOURS)
else:
    last_loaded_ts = None

# ----------------------------------------------------------
# Read Bronze Incrementally
# ----------------------------------------------------------
bronze_df = spark.read.table(bronze_table)

if last_loaded_ts:
    bronze_df = bronze_df.filter(F.col("ingestion_ts") > last_loaded_ts)

# ----------------------------------------------------------
# Standardize & Select Region Grain
# ----------------------------------------------------------
region_df = (
    bronze_df
    .select(
        F.coalesce(F.lower(F.trim(F.col("country"))), F.lit("unknown")).alias("country"),
        F.coalesce(F.lower(F.trim(F.col("state"))), F.lit("unknown")).alias("state"),
        F.coalesce(F.lower(F.trim(F.col("city"))), F.lit("unknown")).alias("city"),
        F.lpad(
            F.regexp_replace(F.col("postal_code").cast("string"), "[^0-9]", ""),
            5,
            "0"
        ).alias("postal_code"),
        F.coalesce(F.lower(F.trim(F.col("region"))), F.lit("unknown")).alias("region"),
        F.col("ingestion_ts")
    )
)

# ----------------------------------------------------------
# De-duplicate (Latest record wins per grain)
# ----------------------------------------------------------
dedup_window = (
    Window
    .partitionBy("country", "state", "city", "postal_code")
    .orderBy(F.col("ingestion_ts").desc())
)

staged_df = (
    region_df
    .withColumn("rn", F.row_number().over(dedup_window))
    .filter(F.col("rn") == 1)
    .drop("rn")
    .withColumn("load_timestamp", F.current_timestamp())
    .withColumn("batch_id", F.lit(BATCH_ID))
)

# ----------------------------------------------------------
# Capture MERGE Metrics (Insert Count)
# ----------------------------------------------------------
history_df = silver_delta.history(1)

metrics = (
    history_df
    .select("operationMetrics")
    .collect()[0]["operationMetrics"]
)

inserted_rows = int(metrics.get("numTargetRowsInserted", 0))
# ----------------------------------------------------------
# Create Table (First Run)
# ----------------------------------------------------------
if not spark.catalog.tableExists(silver_table_dim_region):
    (
        staged_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(silver_table_dim_region)
    )

# ----------------------------------------------------------
# Merge (Type 1 Upsert)
# ----------------------------------------------------------
silver_delta = DeltaTable.forName(spark, silver_table_dim_region)

(
    silver_delta.alias("t")
    .merge(
        staged_df.alias("s"),
        """
        t.country = s.country
        AND t.state = s.state
        AND t.city = s.city
        AND t.postal_code = s.postal_code
        """
    )
    .whenMatchedUpdate(
        set={
            "region": "s.region",
            "load_timestamp": "s.load_timestamp",
            "batch_id": "s.batch_id"
        }
    )
    .whenNotMatchedInsert(
        values={
            "country": "s.country",
            "state": "s.state",
            "city": "s.city",
            "postal_code": "s.postal_code",
            "region": "s.region",
            "load_timestamp": "s.load_timestamp",
            "batch_id": "s.batch_id"
        }
    )
    .execute()
)

# ----------------------------------------------------------
# Logging
# ----------------------------------------------------------
print("dim_region load completed successfully")
print(f"Records inserted in this run : {inserted_rows}")
print(f"batch_id : {BATCH_ID}")


In [0]:
%sql
select count(*) FROM silver_dev.global_mart_retail.dim_region 

In [0]:
%sql
SELECT
    country,
    state,
    city,
    postal_code,
    COUNT(*) AS cnt
FROM silver_dev.global_mart_retail.dim_region
GROUP BY
    country,
    state,
    city,
    postal_code
HAVING COUNT(*) > 1;


In [0]:
%sql
SELECT
    COUNT(*) AS bad_records
FROM silver_dev.global_mart_retail.dim_region
WHERE country IS NULL
   OR state IS NULL
   OR city IS NULL
   OR postal_code IS NULL
   OR region IS NULL;
